<a href="https://colab.research.google.com/github/roshanjain379ux/DS-TRAINING-ROSHANJAIN/blob/main/DS_proj5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce"
)

print("Dataset Path:")
print(path)


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset Path:
/kaggle/input/brazilian-ecommerce


In [ ]:
import pandas as pd

orders_path = "/kaggle/input/brazilian-ecommerce/olist_orders_dataset.csv"

customers_path = "/kaggle/input/brazilian-ecommerce/olist_customers_dataset.csv"

order_items_path = "/kaggle/input/brazilian-ecommerce/olist_order_items_dataset.csv"

products_path = "/kaggle/input/brazilian-ecommerce/olist_products_dataset.csv"

orders_df = pd.read_csv(orders_path)

customers_df = pd.read_csv(customers_path)

order_items_df = pd.read_csv(order_items_path)

products_df = pd.read_csv(products_path)

print("===================================")
print("DATASETS LOADED SUCCESSFULLY")
print("===================================")

print("\nOrders Shape:", orders_df.shape)
print("Customers Shape:", customers_df.shape)
print("Order Items Shape:", order_items_df.shape)
print("Products Shape:", products_df.shape)

master_df = pd.merge(
    orders_df,
    customers_df,
    on='customer_id',
    how='inner'
)

master_df = pd.merge(
    master_df,
    order_items_df,
    on='order_id',
    how='inner'
)

master_df = pd.merge(
    master_df,
    products_df,
    on='product_id',
    how='left'
)

print("\n===================================")
print("MASTER DATAFRAME CREATED")
print("===================================")

print(master_df.head())

master_df['total_price'] = (
    master_df['price']
    +
    master_df['freight_value']
)

customer_revenue = (
    master_df
    .groupby('customer_unique_id')['total_price']
    .sum()
    .reset_index()
    .sort_values(
        by='total_price',
        ascending=False
    )
)

print("\n===================================")
print("TOTAL REVENUE PER CUSTOMER")
print("===================================")

print(customer_revenue.head(10))

customer_order_count = (
    master_df
    .groupby('customer_unique_id')['order_id']
    .nunique()
    .reset_index()
)

customer_order_count.columns = [
    'customer_unique_id',
    'total_orders'
]

single_order_customers = (
    customer_order_count[
        customer_order_count['total_orders'] == 1
    ]
)

print("\n===================================")
print("CUSTOMERS WHO NEVER PLACED SECOND ORDER")
print("===================================")

print(single_order_customers.head())

print("\nTotal Customers With One Order:")
print(len(single_order_customers))

master_df['order_purchase_timestamp'] = pd.to_datetime(
    master_df['order_purchase_timestamp']
)

master_df['order_year'] = (
    master_df['order_purchase_timestamp']
    .dt.year
)

master_df['order_month'] = (
    master_df['order_purchase_timestamp']
    .dt.month
)

print("\n===================================")
print("YEAR AND MONTH EXTRACTION")
print("===================================")

print(
    master_df[
        [
            'order_purchase_timestamp',
            'order_year',
            'order_month'
        ]
    ].head()
)

monthly_sales = (
    master_df
    .groupby(
        [
            'order_year',
            'order_month'
        ]
    )['total_price']
    .sum()
    .reset_index()
)

pivot_sales = pd.pivot_table(
    monthly_sales,
    values='total_price',
    index='order_year',
    columns='order_month',
    aggfunc='sum'
)

print("\n===================================")
print("PIVOT TABLE")
print("===================================")

print(pivot_sales)

melted_sales = pd.melt(
    pivot_sales.reset_index(),
    id_vars='order_year',
    var_name='Month',
    value_name='Sales'
)

print("\n===================================")
print("MELTED DATAFRAME")
print("===================================")

print(melted_sales.head(15))

print("\n===================================")
print("PROJECT COMPLETED SUCCESSFULLY")
print("===================================")

DATASETS LOADED SUCCESSFULLY

Orders Shape: (99441, 8)
Customers Shape: (99441, 5)
Order Items Shape: (112650, 7)
Products Shape: (32951, 9)

MASTER DATAFRAME CREATED
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_d